## Imports

In [1]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
import numpy as np
import matplotlib.pyplot as plt
import os

I0000 00:00:1778335371.283443   12289 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1778335371.327830   12289 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1778335372.822756   12289 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


## Utils

In [2]:
def build_discriminator():
    model = models.Sequential(name="Discriminator")
    
    model.add(layers.Conv2D(64, (5, 5), strides=(2, 2), padding='same', input_shape=[28, 28, 1]))
    model.add(layers.LeakyReLU())
    model.add(layers.Dropout(0.3))

    model.add(layers.Conv2D(128, (5, 5), strides=(2, 2), padding='same'))
    model.add(layers.LeakyReLU())
    model.add(layers.Dropout(0.3))

    model.add(layers.Flatten())
    model.add(layers.Dense(1, activation='sigmoid')) # Sigmoid to output probability (real vs fake)
    return model

In [3]:
def build_generator(latent_dim):
    model = models.Sequential(name="Generator")
    
    # Base dense layer to process the random noise vector
    model.add(layers.Dense(7 * 7 * 256, use_bias=False, input_shape=(latent_dim,)))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())
    model.add(layers.Reshape((7, 7, 256)))
    
    # Conv2DTranspose is used for upsampling characteristics [cite: 342, 345]
    # Upsample to 7x7
    model.add(layers.Conv2DTranspose(128, (5, 5), strides=(1, 1), padding='same', use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())

    # Upsample to 14x14
    model.add(layers.Conv2DTranspose(64, (5, 5), strides=(2, 2), padding='same', use_bias=False))
    model.add(layers.BatchNormalization())
    model.add(layers.LeakyReLU())

    # Final Upsample to 28x28x1
    # Tanh activation is used at the output layer [cite: 344]
    model.add(layers.Conv2DTranspose(1, (5, 5), strides=(2, 2), padding='same', use_bias=False, activation='tanh'))
    return model

In [4]:
class DCGAN(models.Model):
    def __init__(self, discriminator, generator, latent_dim):
        super(DCGAN, self).__init__()
        self.discriminator = discriminator
        self.generator = generator
        self.latent_dim = latent_dim

    def compile(self, d_optimizer, g_optimizer, loss_fn):
        super(DCGAN, self).compile()
        self.d_optimizer = d_optimizer
        self.g_optimizer = g_optimizer
        self.loss_fn = loss_fn
        self.d_loss_metric = tf.keras.metrics.Mean(name="d_loss")
        self.g_loss_metric = tf.keras.metrics.Mean(name="g_loss")

    @property
    def metrics(self):
        return [self.d_loss_metric, self.g_loss_metric]

    # Custom training step calculating both generator and discriminator losses [cite: 336]
    def train_step(self, real_images):
        batch_size = tf.shape(real_images)[0]
        
        # 1. Train the Discriminator
        random_latent_vectors = tf.random.normal(shape=(batch_size, self.latent_dim))
        generated_images = self.generator(random_latent_vectors)
        combined_images = tf.concat([generated_images, real_images], axis=0)
        
        # Labels: 0 for fake, 1 for real
        labels = tf.concat([tf.zeros((batch_size, 1)), tf.ones((batch_size, 1))], axis=0)
        # Trick: Add slight random noise to labels for better stability
        labels += 0.05 * tf.random.uniform(tf.shape(labels))

        with tf.GradientTape() as tape:
            predictions = self.discriminator(combined_images)
            d_loss = self.loss_fn(labels, predictions)
            
        grads = tape.gradient(d_loss, self.discriminator.trainable_weights)
        self.d_optimizer.apply_gradients(zip(grads, self.discriminator.trainable_weights))

        # 2. Train the Generator
        random_latent_vectors = tf.random.normal(shape=(batch_size, self.latent_dim))
        misleading_labels = tf.ones((batch_size, 1)) # We want the discriminator to think these are real

        with tf.GradientTape() as tape:
            predictions = self.discriminator(self.generator(random_latent_vectors))
            g_loss = self.loss_fn(misleading_labels, predictions)
            
        grads = tape.gradient(g_loss, self.generator.trainable_weights)
        self.g_optimizer.apply_gradients(zip(grads, self.generator.trainable_weights))

        # Update metrics
        self.d_loss_metric.update_state(d_loss)
        self.g_loss_metric.update_state(g_loss)
        return {"d_loss": self.d_loss_metric.result(), "g_loss": self.g_loss_metric.result()}

## Dataset

In [5]:
(X_train, _), (_, _) = tf.keras.datasets.fashion_mnist.load_data()

# Normalize data to the range [-1, 1] for the Generator's tanh activation
X_train = X_train.astype('float32')
X_train = (X_train - 127.5) / 127.5 
X_train = np.expand_dims(X_train, axis=-1) # Shape becomes (60000, 28, 28, 1)

BATCH_SIZE = 256
LATENT_DIM = 100

# Create a TensorFlow Dataset object
dataset = tf.data.Dataset.from_tensor_slices(X_train).shuffle(60000).batch(BATCH_SIZE)

29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 2us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 1s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


E0000 00:00:1778335377.409399   12289 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1778335377.426035   12289 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


## Training

In [6]:
discriminator = build_discriminator()
generator = build_generator(LATENT_DIM)

gan = DCGAN(discriminator=discriminator, generator=generator, latent_dim=LATENT_DIM)

# Use Adam optimizer and Binary Crossentropy as per the laboratory logic [cite: 332]
gan.compile(
    d_optimizer=tf.keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5),
    g_optimizer=tf.keras.optimizers.Adam(learning_rate=0.0002, beta_1=0.5),
    loss_fn=tf.keras.losses.BinaryCrossentropy()
)

# Callback 1: Early Stopping
# Note: Early stopping in GANs is tricky because losses oscillate. 
# We monitor generator loss, stopping if it stops improving (plateaus heavily).
early_stop = callbacks.EarlyStopping(
    monitor='g_loss',
    patience=10, 
    restore_best_weights=False 
)

# Callback 2: Custom Save Method and Image Visualizer
class SaveModelAndImages(callbacks.Callback):
    def on_epoch_end(self, epoch, logs=None):
        # Save the Generator Model at specific intervals
        if (epoch + 1) % 10 == 0:
            os.makedirs('saved_models', exist_ok=True)
            # Saving the model using the standard save method
            self.model.generator.save(f"saved_models/generator_epoch_{epoch+1}.h5")
            print(f"\\nSaved generator model at epoch {epoch+1}")
            
        # Print intermediate visual results occasionally 
        if (epoch + 1) % 5 == 0:
            noise = tf.random.normal([1, LATENT_DIM])
            gen_image = self.model.generator(noise, training=False)
            plt.imshow(gen_image[0, :, :, 0] * 127.5 + 127.5, cmap='gray')
            plt.title(f"Epoch {epoch+1}")
            plt.axis('off')
            plt.show()

# Train the Model
EPOCHS = 50
gan.fit(
    dataset, 
    epochs=EPOCHS, 
    callbacks=[early_stop, SaveModelAndImages()]
)

# Final definitive save method for the completed model
generator.save('final_generator_model.keras')
print("Training complete and final model saved successfully.")

Epoch 1/50


/home/grenadinu/University/Neural-Networks/.venv/lib/python3.11/site-packages/keras/src/layers/convolutional/base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
/home/grenadinu/University/Neural-Networks/.venv/lib/python3.11/site-packages/keras/src/layers/core/dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


 35/235 ━━━━━━━━━━━━━━━━━━━━ 2:49 847ms/step - d_loss: 0.5246 - g_loss: 0.7873

KeyboardInterrupt: 